# cn_a_big_money_rotation v0.1 — verdict 可视化分析

对应设计: `GinkgoRoad/docs/GinkgoBrain/A股日频-大单净流入横截面轮动-设计.md`

verdict 报告: NO-ALPHA. 本 notebook 用 7 张图验证 verdict, 看是否能从数据里挖出反驳证据.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

REPORTS = Path('/home/davidlyu/projects/GinkgoBrain/reports/v0.1_20260513_174535')

metrics = json.loads((REPORTS / 'metrics.json').read_text())
equity = pd.read_csv(REPORTS / 'equity.csv', parse_dates=['trade_date']).set_index('trade_date')
daily_ic = pd.read_csv(REPORTS / 'daily_ic.csv').set_index('day_idx')
diag_path = REPORTS / 'daily_diagnostics.csv'
if diag_path.exists():
    diag = pd.read_csv(diag_path, parse_dates=['trade_date']).set_index('trade_date')
else:
    diag = None

print('verdict:', metrics['verdict'])
print('verdict_notes:', metrics['verdict_notes'])
print('extra_notes:', metrics['extra_notes'])
print()
print('IC_A: mean=%.4f  t-stat=%.3f  n=%d' % (
    metrics['metrics']['ic_a']['mean'],
    metrics['metrics']['ic_a']['t_stat'],
    metrics['metrics']['ic_a']['n']))
print('IC_B: mean=%.4f  t-stat=%.3f  n=%d' % (
    metrics['metrics']['ic_b']['mean'],
    metrics['metrics']['ic_b']['t_stat'],
    metrics['metrics']['ic_b']['n']))

## §1 净值曲线 vs BM2 等权全市场

策略 equity (起点标 1.0) vs BM2 等权 universe（含 0.01%/日 摩擦）.
**预期**：策略远跑输 BM2（年化差 ~40%）.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
eq_norm = equity['equity'] / equity['equity'].iloc[0]
ax.plot(eq_norm.index, eq_norm.values, label='Strategy (cn_a_big_money_rotation)', linewidth=1.6)
ax.plot(equity.index, equity['bm2_equity'], label='BM2 (Equal-Weighted Universe)', linewidth=1.6, alpha=0.85)
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Equity Curve: Strategy vs BM2 (normalized to 1.0)')
ax.set_xlabel('Trade Date')
ax.set_ylabel('Normalized Equity')
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

## §2 月度收益对比

月度策略 vs BM2 vs 超额（差值）.

In [ ]:
monthly_strat = (equity['equity'] / equity['equity'].iloc[0]).resample('ME').last().pct_change().fillna(0)
monthly_bm2 = equity['bm2_equity'].resample('ME').last().pct_change().fillna(0)
monthly_excess = monthly_strat - monthly_bm2

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(monthly_strat))
width = 0.3
ax.bar(x - width, monthly_strat.values, width, label='Strategy', alpha=0.85)
ax.bar(x,         monthly_bm2.values,   width, label='BM2',      alpha=0.85)
ax.bar(x + width, monthly_excess.values, width, label='Excess (Strat-BM2)', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Monthly Returns')
ax.set_xticks(x)
ax.set_xticklabels([d.strftime('%Y-%m') for d in monthly_strat.index], rotation=45)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## §3 日度 IC 时序

T 日 score 与 T+4 累计相对收益的 Spearman IC. **预期**：在 0 附近随机飘动.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_ic.index, daily_ic['ic_a'], label='IC_A (big_net_per_mv)', alpha=0.6, linewidth=0.8)
ax.plot(daily_ic.index, daily_ic['ic_b'], label='IC_B (big_net_ratio)',  alpha=0.6, linewidth=0.8)
ax.plot(daily_ic.index, daily_ic['ic_a'].rolling(20).mean(), label='IC_A 20d MA', color='C0', linewidth=2.2)
ax.plot(daily_ic.index, daily_ic['ic_b'].rolling(20).mean(), label='IC_B 20d MA', color='C1', linewidth=2.2)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(0.02, color='green', linestyle='--', alpha=0.6, label='门槛 0.02')
ax.set_title('Daily IC Time Series (Spearman vs T+4 forward relative return)')
ax.set_xlabel('day_idx')
ax.set_ylabel('IC')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §4 IC 分布直方图

判断 IC 是否近似 N(0, σ²) —— 如果是，说明信号纯噪声.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, ['ic_a', 'ic_b'], ['C0', 'C1']):
    data = daily_ic[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.6, edgecolor='black')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(data.mean(), color='red', linewidth=2, label=f'mean={data.mean():.4f}')
    ax.axvline(0.02, color='green', linestyle='--', alpha=0.7, label='门槛 0.02')
    ax.set_title(f'{col} distribution (mean={data.mean():.4f}, std={data.std():.4f})')
    ax.set_xlabel('Daily IC')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §5 Drawdown 曲线

In [ ]:
def drawdown(s):
    return (s / s.cummax() - 1.0) * 100

eq_norm = equity['equity'] / equity['equity'].iloc[0]
dd_strat = drawdown(eq_norm)
dd_bm2 = drawdown(equity['bm2_equity'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(dd_strat.index, dd_strat.values, 0, color='C0', alpha=0.3, label='Strategy DD')
ax.fill_between(dd_bm2.index, dd_bm2.values, 0, color='C1', alpha=0.3, label='BM2 DD')
ax.plot(dd_strat.index, dd_strat.values, color='C0', linewidth=1)
ax.plot(dd_bm2.index, dd_bm2.values, color='C1', linewidth=1)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Drawdown (%)')
ax.set_xlabel('Trade Date')
ax.set_ylabel('DD (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §6 累积 IC（cumulative sum of daily IC）

若信号有 alpha，累积 IC 应单调上升；若纯噪声，应在 0 附近随机游走.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_ic.index, daily_ic['ic_a'].fillna(0).cumsum(), label='cumsum IC_A')
ax.plot(daily_ic.index, daily_ic['ic_b'].fillna(0).cumsum(), label='cumsum IC_B')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Cumulative Daily IC')
ax.set_xlabel('day_idx')
ax.set_ylabel('cumsum')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §7 Diagnostics: 持仓笔数 / cash / 双信号 overlap 时序

如果 daily_diagnostics.csv 不存在（旧 backtest 跑的），本节跳过.

In [ ]:
if diag is not None:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(diag.index, diag['n_positions'], label='n_positions', color='C2')
    axes[0].plot(diag.index, diag['selected_size'], label='selected_size (Top-N)', color='C0', alpha=0.6)
    axes[0].plot(diag.index, diag['universe_size'] / 100, label='universe_size / 100', color='gray', alpha=0.6)
    axes[0].set_title('Position Count vs Selected vs Universe (/100)')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    cash_pct = diag['cash'] / diag['equity']
    axes[1].plot(diag.index, cash_pct * 100, color='C3')
    axes[1].set_title('Cash Ratio (%)')
    axes[1].set_ylabel('% of equity')
    axes[1].grid(alpha=0.3)

    axes[2].plot(diag.index, diag['overlap_ab'] * 100, color='C4', alpha=0.6, linewidth=0.8)
    axes[2].plot(diag.index, (diag['overlap_ab'] * 100).rolling(20).mean(), color='C4', linewidth=2.2, label='20d MA')
    axes[2].axhline(60, color='green', linestyle='--', alpha=0.7, label='门槛 60%')
    axes[2].set_title('Dual Signal Overlap (A vs B Top-N) %')
    axes[2].legend(); axes[2].grid(alpha=0.3)
    axes[2].set_xlabel('Trade Date')
    plt.tight_layout()
    plt.show()
else:
    print('daily_diagnostics.csv 不存在 → 跳过 §7（请用 M6 更新后的 backtest 重跑）')

## §8 解读 & 决策

**逐图答 4 个问题**：

1. **§3/§4/§6 IC 是否真的是噪声？** 如果 IC 在 0 附近近似 N(0, σ²)、cumsum 随机游走 → 信号无方向性；如果 IC 有阶段性偏正/偏负 → 可能分段有效。
2. **§5 策略 DD 形态？** 是平稳下行（信号始终错），还是某段集中亏（特定市场状态下错）。
3. **§7 持仓笔数 / cash 比例** 是否长期 underexposed？如果 cash 长期 > 20%，部分 alpha 损失来自没满仓；信号本身可能有效但策略机制吃了 alpha。
4. **§7 overlap_ab** 时序：是恒定低还是某段高某段低？如果某段 > 60% → 那段两个信号有共识，可能有 alpha 窗口期。

### 三类发现路径

- **路径 A**: §3/§4/§6 完全噪声 + §5 平稳下行 + §7 cash 不超 20% → **信号确认无 alpha**，关闭。
- **路径 B**: §3 IC 有时段性强相关 + §7 overlap 时段性高 → **alpha 是 regime-dependent**，需要先识别 regime（v0.2 方向）。
- **路径 C**: §3/§4/§6 噪声 但 §7 cash 长期 > 30% → 信号可能有效但**策略机制 underexposure 吃了 alpha**，应该把容量约束 / 持有期重新调（v0.1.5 工程优化方向）。